# 🏥 GI VQA — Conversational Chat Interface
**Ask unlimited questions about any GI endoscopy image — like ChatGPT**

## ⚙️ Cell 1 — Setup (run once)

In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')
os.chdir(os.path.expanduser('~'))
for p in [os.path.expanduser('~'), os.path.expanduser('~/vqa_gi_thesis/src')]:
    if p not in sys.path: sys.path.insert(0, p)

import torch, numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, HTML, Image as IPImage, clear_output
import ipywidgets as widgets

print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')
print('✅ Setup complete')

Device: CUDA
✅ Setup complete


## 🔄 Cell 2 — Load Pipeline (run once)

In [2]:
from explainability import ExplainabilityPredictor, plot_explainability_report
print('Loading pipeline ... (~30 seconds)')
predictor = ExplainabilityPredictor()

chat_state = {'image_path': None, 'history': [],
              'last_result': None, 'disease_vec': None}

DISEASE_NAMES = [
    'Polyp (Pedunculated)','Polyp (Sessile)','Polyp (Hyperplastic)',
    'Esophagitis','Gastritis','Ulcerative Colitis',"Crohn's Disease",
    "Barrett's Esophagus",'Gastric Ulcer','Duodenal Ulcer',
    'Erosion','Hemorrhoids','Diverticulosis','Normal Cecum',
    'Normal Pylorus','Normal Z-Line','Ileocecal Valve',
    'Retroflex Rectum','Retroflex Stomach','Dyed Lifted Polyp',
    'Dyed Resection Margin','Foreign Body','Instrument'
]
QTYPE_COLORS = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336','#00BCD4']

# ── Preset questions grouped by type ──────────────────────────────────
PRESET_QUESTIONS = {
    '✅ Yes / No': [
        'Is there a polyp visible?',
        'Is this a normal image?',
        'Are there any instruments visible?',
        'Is there any abnormality present?',
        'Is there evidence of inflammation?',
        'Are there any foreign bodies visible?',
        'Is there any bleeding visible?',
        'Has the polyp been removed?',
    ],
    '🎨 Color': [
        'What color is the finding?',
        'What color is the polyp?',
        'What is the color of the lesion?',
        'What color is the mucosa?',
    ],
    '📍 Location': [
        'Where is the finding located?',
        'Where is the polyp located in the image?',
        'Which part of the GI tract is shown?',
        'Where is the abnormality in the image?',
        'What anatomical region is visible?',
    ],
    '🔢 Count': [
        'How many polyps are visible?',
        'How many findings are present?',
        'How many instruments are visible?',
        'How many abnormalities can you see?',
    ],
    '📋 Single Choice': [
        'What type of polyp is present?',
        'What procedure is being performed?',
        'What is the diagnosis?',
        'What type of lesion is visible?',
        'What instrument is visible?',
        'What anatomical landmark is present?',
        'What is the size of the polyp?',
        'What abnormality is most prominent?',
    ],
    '📝 Multiple Choice': [
        'What abnormalities are present in the image?',
        'What findings are visible in this image?',
        'Describe all visible elements in the image.',
        'What instruments and findings are present?',
    ],
}

print('\n🚀 Pipeline ready! Run Cell 3.')

✅  Imports from preprocessing / stage1 / stage2 successful
✅  Imports from preprocessing / stage1 / stage3 successful
Loading pipeline ... (~30 seconds)
🔍  Loading ExplainabilityPredictor ...
✅  TextPreprocessor ready  |  vocab=30,522  max_len=128
🔒  Backbone frozen — 0 trainable backbone parameters
🧠  TreeNet  |  total=24,177,495  trainable=669,463  frozen=23,508,032
   GradCAM target: backbone[7][-1].conv3  →  Conv2d  out_ch=2048
🔒  Backbone frozen — 0 trainable backbone parameters
🧠  TreeNet  |  total=24,177,495  trainable=669,463  frozen=23,508,032
🔒  FrozenVisualEncoder loaded  (stage1 best_f1=0.9925)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: ./checkpoints/best_model
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔒  FrozenTextEncoder loaded  (stage2 DistilBERT)

🧠  Stage3MultimodalFusion
    Total params     : 93,734,493
    Trainable        : 3,194,118  (CrossAttn + DiseaseGate + FusionMLP)
    Frozen           : 90,540,375  (ResNet50 + DistilBERT)
✅  FusionExtractor loaded  (best_acc=0.9250)

🧠  Stage4AnswerGenerator
    6 specialised heads  |  Total params: 925,827
    [yn      ] →    2 classes  |  138,242 params
    [single  ] →  200 classes  |  189,128 params
    [multi   ] →  150 classes  |  176,278 params
    [color   ] →   12 classes  |  140,812 params
    [loc     ] →   15 classes  |  141,583 params
    [count   ] →    8 classes  |  139,784 params
✅  ExplainabilityPredictor ready


🚀 Pipeline ready! Run Cell 3.


## 💬 Cell 3 — Chat Interface with Question Presets

In [3]:
# ── Image loader widgets ───────────────────────────────────────────────
uploader     = widgets.FileUpload(accept='image/*', multiple=False,
                                   description='📸 Upload',
                                   layout=widgets.Layout(width='130px'))
img_path_box = widgets.Text(
    placeholder='or paste image path: ./data/kvasir_raw/images/xxx.jpg',
    layout=widgets.Layout(width='480px'))
load_btn     = widgets.Button(description='Load Image', button_style='info',
                               layout=widgets.Layout(width='110px'))
random_btn   = widgets.Button(description='🎲 Random', button_style='',
                               layout=widgets.Layout(width='100px'))
status_lbl   = widgets.HTML(value='<i style="color:gray">No image loaded</i>')

# ── Question type selector ─────────────────────────────────────────────
type_selector = widgets.ToggleButtons(
    options=list(PRESET_QUESTIONS.keys()),
    description='',
    button_style='',
    layout=widgets.Layout(width='100%'),
    style={'button_width': '150px'}
)

# ── Preset question buttons (dynamic) ─────────────────────────────────
preset_out = widgets.Output()

def render_preset_buttons(qtype):
    with preset_out:
        clear_output(wait=True)
        questions = PRESET_QUESTIONS[qtype]
        btns = []
        for q in questions:
            b = widgets.Button(
                description=q[:55],
                layout=widgets.Layout(width='98%', height='32px'),
                style={'button_color': '#E3F2FD',
                       'font_weight': 'normal'}
            )
            b._question = q   # store full question
            def on_preset_click(btn):
                q_input.value = btn._question
                ask_question(btn._question)
            b.on_click(on_preset_click)
            btns.append(b)
        display(widgets.VBox(btns,
            layout=widgets.Layout(
                border='1px solid #ddd',
                padding='6px',
                border_radius='8px',
                background_color='#f9f9f9'
            )))

def on_type_change(change):
    render_preset_buttons(change['new'])

type_selector.observe(on_type_change, names='value')

# ── Custom question input ──────────────────────────────────────────────
q_input    = widgets.Text(
    placeholder='Or type your own question here and press Enter ...',
    layout=widgets.Layout(width='560px'))
ask_btn    = widgets.Button(description='Ask ▶', button_style='success',
                             layout=widgets.Layout(width='90px'))
explain_btn= widgets.Button(description='🔬 GradCAM', button_style='warning',
                             layout=widgets.Layout(width='110px'))
clear_btn  = widgets.Button(description='🗑 Clear', button_style='danger',
                             layout=widgets.Layout(width='90px'))

# ── Output areas ───────────────────────────────────────────────────────
chat_out  = widgets.Output()
image_out = widgets.Output()


# ── Helpers ────────────────────────────────────────────────────────────
def render_message(role, text, extra=''):
    bg     = '#e8f5e9' if role.startswith('AI') else '#e3f2fd'
    border = '#4CAF50' if role.startswith('AI') else '#2196F3'
    align  = 'left'   if role.startswith('AI') else 'right'
    return (f'<div style="display:flex;justify-content:{align};margin:5px 0">'
            f'<div style="max-width:90%;padding:9px 13px;background:{bg};'
            f'border-left:4px solid {border};border-radius:8px;'
            f'font-family:sans-serif">'
            f'<b style="color:{border};font-size:12px">{role}</b><br>'
            f'<span style="color:#1a1a2e;font-size:14px">{text}</span>'
            f'{extra}</div></div>')

def render_answer(result):
    r, conf = result['route'], result['route_conf']*100
    ans  = result['answer'].upper() if result['answer'] else '&lt;OUT OF VOCAB&gt;'
    rname = result['route_name']
    d_vec  = result['disease_vec']
    color  = QTYPE_COLORS[r]
    cc     = '#4CAF50' if conf>=80 else '#FF9800' if conf>=50 else '#F44336'
    active = sorted([(DISEASE_NAMES[i], d_vec[i])
                     for i in range(23) if d_vec[i]>0.35],
                    key=lambda x:-x[1])
    # Disease pills
    dis_pills = ''.join(
        f'<span style="background:#E3F2FD;border:1px solid #90CAF9;'
        f'border-radius:4px;padding:1px 6px;margin:2px;font-size:11px">'
        f'🔬 {n} {p*100:.0f}%</span>'
        for n, p in active[:4])
    dis_html = f'<br>{dis_pills}' if active else ''
    # Explanation — parse from result
    expl = result.get('explanation', '')
    # Extract just the key lines — not the full block
    expl_lines = []
    for line in expl.split('\n'):
        line = line.strip()
        if line.startswith('The question was classified'):
            expl_lines.append(line)
        elif line.startswith('Route') and 'head selected' in line:
            expl_lines.append(line)
        elif any(line.startswith(x) for x in [
            'Binary','Single-choice','Multi-label',
            'Color classification','Anatomical','Ordinal',
            'High disease','Low disease']):
            expl_lines.append(line)
    expl_html = ''
    if expl_lines:
        expl_html = ('<br><div style="margin-top:6px;padding:7px 10px;'
                     'background:#FFFDE7;border-left:3px solid #FFC107;'
                     'border-radius:4px;font-size:11px;color:#555">'
                     '<b>💡 Explanation:</b><br>'
                     + '<br>'.join(expl_lines[:3])
                     + '</div>')
    extra = (f'<br><span style="font-size:11px;color:#777">'
             f'Type: <b style="color:{color}">{rname}</b> &nbsp;|&nbsp;'
             f'Confidence: <b style="color:{cc}">{conf:.1f}%</b></span>'
             f'{dis_html}{expl_html}')
    return render_message('AI 🤖', ans, extra=extra)

def refresh_chat():
    with chat_out:
        clear_output(wait=True)
        html = ('<div style="height:380px;overflow-y:auto;padding:10px;'
                'background:#fafafa;border:1px solid #ddd;border-radius:10px">')
        if not chat_state['history']:
            html += ('<p style="color:#bbb;text-align:center;margin-top:160px">'
                     'Load an image, pick a question type, then click a question ...</p>')
        for item in chat_state['history']:
            if 'info' in item:
                html += render_message('System ⚙️', item['info'])
            else:
                html += render_message('You 👤', item['question'])
                if 'result' in item:
                    html += render_answer(item['result'])
        html += '</div>'
        display(HTML(html))

def show_image(path):
    with image_out:
        clear_output(wait=True)
        img = PILImage.open(path).convert('RGB')
        fig, ax = plt.subplots(figsize=(3.8, 3.8))
        ax.imshow(img); ax.axis('off')
        ax.set_title(os.path.basename(path)[:28], fontsize=8, pad=3)
        plt.tight_layout(); plt.show()
        # Disease bar
        if chat_state['disease_vec'] is not None:
            dv = chat_state['disease_vec']
            top = [i for i in np.argsort(dv)[::-1][:6] if dv[i]>0.08]
            if top:
                fig2, ax2 = plt.subplots(figsize=(3.8, 1.8))
                cols = ['#f44336' if dv[i]>=0.5 else
                        '#ff9800' if dv[i]>=0.3 else '#90CAF9' for i in top]
                ax2.barh([DISEASE_NAMES[i][:18] for i in top[::-1]],
                         [dv[i]*100 for i in top[::-1]],
                         color=cols[::-1], edgecolor='white', alpha=0.9)
                ax2.axvline(50, color='red', lw=1, ls='--', alpha=0.4)
                ax2.set_xlim(0,110); ax2.tick_params(labelsize=6)
                ax2.set_title('Disease Context', fontsize=8, fontweight='bold')
                plt.tight_layout(); plt.show()

def load_image(path):
    chat_state['image_path'] = path
    r = predictor.predict_full(path, 'Is there any abnormality?')
    chat_state['disease_vec'] = r['disease_vec']
    chat_state['last_result'] = r
    active = [DISEASE_NAMES[i] for i in range(23) if r['disease_vec'][i]>0.4]
    chat_state['history'].append({'info':
        f"<b>Image loaded:</b> {os.path.basename(path)}<br>"
        f"Findings: {', '.join(active) if active else 'None above 40%'}"})
    status_lbl.value = f'<b style="color:green">✅ {os.path.basename(path)}</b>'
    show_image(path)
    refresh_chat()

def ask_question(question):
    if not chat_state['image_path']:
        status_lbl.value = '<span style="color:orange">⚠️ Load an image first</span>'
        return
    if not question.strip(): return
    status_lbl.value = '<i style="color:gray">🔄 Thinking ...</i>'
    chat_state['history'].append({'question': question})
    refresh_chat()
    result = predictor.predict_full(chat_state['image_path'], question)
    chat_state['last_result'] = result
    chat_state['history'][-1]['result'] = result
    status_lbl.value = f'<b style="color:green">✅ {os.path.basename(chat_state["image_path"])}</b>'
    q_input.value = ''
    refresh_chat()


# ── Button callbacks ───────────────────────────────────────────────────
def on_load(b):
    if uploader.value:
        fname = list(uploader.value.keys())[0]
        fdata = uploader.value[fname]['content']
        path  = f'/tmp/chat_{fname}'
        with open(path,'wb') as f: f.write(fdata)
        load_image(path)
    elif img_path_box.value.strip():
        path = os.path.expanduser(img_path_box.value.strip())
        if os.path.exists(path): load_image(path)
        else: status_lbl.value = f'<span style="color:red">❌ Not found: {path}</span>'
    else:
        status_lbl.value = '<span style="color:orange">⚠️ Upload or paste a path</span>'

def on_random(b):
    import random, glob
    imgs = glob.glob('./data/kvasir_raw/images/*.jpg')
    if imgs: load_image(random.choice(imgs))
    else: status_lbl.value = '<span style="color:red">❌ Images not found</span>'

def on_ask(b):    ask_question(q_input.value)
def on_enter(w):  ask_question(w.value)
def on_explain(b):
    if chat_state['last_result']:
        path = plot_explainability_report(chat_state['last_result'])
        with chat_out:
            clear_output(wait=True)
            display(HTML('<b>📊 GradCAM Explainability Report:</b>'))
            display(IPImage(path, width=860))
        status_lbl.value = f'<b style="color:green">📊 Saved: {path}</b>'
    else:
        status_lbl.value = '<span style="color:orange">⚠️ Ask a question first</span>'

def on_clear(b):
    chat_state['history'] = []
    refresh_chat()

load_btn.on_click(on_load)
random_btn.on_click(on_random)
ask_btn.on_click(on_ask)
q_input.on_submit(on_enter)
explain_btn.on_click(on_explain)
clear_btn.on_click(on_clear)


# ── Render full UI ─────────────────────────────────────────────────────
display(HTML('<h3 style="color:#1a237e;margin-bottom:4px">🏥 GI VQA Chat</h3>'))

# Image loader row
display(HTML('<b style="color:#555">Step 1 — Load an image:</b>'))
display(widgets.HBox([uploader, img_path_box, load_btn, random_btn]))
display(status_lbl)
display(HTML('<hr style="margin:8px 0">'))

# Question type tabs
display(HTML('<b style="color:#555">Step 2 — Pick a question type:</b>'))
display(type_selector)

# Main area: presets + chat + image
display(widgets.HBox([
    widgets.VBox([
        widgets.HTML('<b style="color:#555;font-size:12px">Click a question:</b>'),
        preset_out
    ], layout=widgets.Layout(width='300px', min_width='300px')),
    widgets.VBox([
        widgets.HTML('<b style="color:#555;font-size:12px">Chat:</b>'),
        chat_out
    ], layout=widgets.Layout(width='500px')),
    widgets.VBox([
        widgets.HTML('<b style="color:#555;font-size:12px">Image:</b>'),
        image_out
    ], layout=widgets.Layout(width='280px')),
]))

# Custom input row
display(HTML('<b style="color:#555;font-size:12px">Or type your own:</b>'))
display(widgets.HBox([q_input, ask_btn, explain_btn, clear_btn]))

# Init
render_preset_buttons(list(PRESET_QUESTIONS.keys())[0])
refresh_chat()

HTML(value='<i style="color:gray">No image loaded</i>')

ToggleButtons(layout=Layout(width='100%'), options=('✅ Yes / No', '🎨 Color', '📍 Location', '🔢 Count', '📋 Singl…

## 📊 Cell 4 — Full GradCAM Report

In [ ]:
if chat_state['last_result']:
    path = plot_explainability_report(chat_state['last_result'])
    display(IPImage(path, width=1000))
else:
    print('Ask a question in Cell 3 first')

## 📜 Cell 5 — Conversation History Table

In [ ]:
import pandas as pd
rows = []
for item in chat_state['history']:
    if 'result' in item:
        r = item['result']
        rows.append({'Question': item['question'][:65],
                     'Answer': r['answer'],
                     'Type': r['route_name'],
                     'Confidence': f"{r['route_conf']*100:.1f}%"})
if rows:
    display(HTML(pd.DataFrame(rows).to_html(index=False,
        border=1).replace('<table',
        '<table style="font-size:13px;font-family:monospace;width:100%"')))
else:
    print('No conversation yet')